## Neural Network example
Let's explore implementation of Neural Network!

In [82]:
from typing import List

# define Vector + dot (replaces scratch.linear_algebra)
Vector = List[float]

def dot(v: Vector, w: Vector) -> float:
    return sum(v_i * w_i for v_i, w_i in zip(v, w))


def step_function(x: float) -> float:
    return 1.0 if x >= 0 else 0.0


def perceptron_output(weights: Vector, bias: float, x: Vector) -> float:
    """Returns 1 if the perceptron 'fires', 0 if not"""
    calculation = dot(weights, x) + bias
    return step_function(calculation)


# AND gate
and_weights = [2., 2]
and_bias = -3.

assert perceptron_output(and_weights, and_bias, [1, 1]) == 1
assert perceptron_output(and_weights, and_bias, [0, 1]) == 0
assert perceptron_output(and_weights, and_bias, [1, 0]) == 0
assert perceptron_output(and_weights, and_bias, [0, 0]) == 0


# OR gate
or_weights = [2., 2]
or_bias = -1.

assert perceptron_output(or_weights, or_bias, [1, 1]) == 1
assert perceptron_output(or_weights, or_bias, [0, 1]) == 1
assert perceptron_output(or_weights, or_bias, [1, 0]) == 1
assert perceptron_output(or_weights, or_bias, [0, 0]) == 0


# NOT gate
not_weights = [-2.]
not_bias = 1.

assert perceptron_output(not_weights, not_bias, [0]) == 1
assert perceptron_output(not_weights, not_bias, [1]) == 0

print("Perceptron tests passed!")

Perceptron tests passed!


### Sigmoid function and feedforward


In [83]:
import math

def sigmoid(t: float) -> float:
    return 1 / (1 + math.exp(-t))

def neuron_output(weights: Vector, inputs: Vector) -> float:
    # weights includes the bias term, inputs includes a 1
    return sigmoid(dot(weights, inputs))

from typing import List

def feed_forward(neural_network: List[List[Vector]],
                 input_vector: Vector) -> List[Vector]:
    """
    Feeds the input vector through the neural network.
    Returns the outputs of all layers (not just the last one).
    """
    outputs: List[Vector] = []

    for layer in neural_network:
        input_with_bias = input_vector + [1]              # Add a constant.
        output = [neuron_output(neuron, input_with_bias)  # Compute the output
                  for neuron in layer]                    # for each neuron.
        outputs.append(output)                            # Add to results.

        # Then the input to the next layer is the output of this one
        input_vector = output

    return outputs


### Example
Build the XOR gate that we couldn’t build with a single perceptron. 

In [84]:
# and_gate = min
# or_gate = max
# xor_gate = lambda x, y: 0 if x == y else 1 

xor_network = [# hidden layer
               [[20., 20, -30],      # 'and' neuron
                [20., 20, -10]],     # 'or'  neuron
               # output layer
               [[-60., 60, -30]]]    # '2nd input but not 1st input' neuron

# feed_forward returns the outputs of all layers, so the [-1] gets the
# final output, and the [0] gets the value out of the resulting vector
assert 0.000 < feed_forward(xor_network, [0, 0])[-1][0] < 0.001
assert 0.999 < feed_forward(xor_network, [1, 0])[-1][0] < 1.000
assert 0.999 < feed_forward(xor_network, [0, 1])[-1][0] < 1.000
assert 0.000 < feed_forward(xor_network, [1, 1])[-1][0] < 0.001

In [85]:
from IPython.display import Image
from IPython.core.display import HTML 
Image(url= "https://learning.oreilly.com/library/view/data-science-from/9781492041122/assets/dsf2_1803.png")


### Backpropagation
we use data to train neural networks. The typical approach is an algorithm called backpropagation, which uses gradient descent or one of its variants.

In [86]:
def sqerror_gradients(network: List[List[Vector]],
                      input_vector: Vector,
                      target_vector: Vector) -> List[List[Vector]]:
    """
    Given a neural network, an input vector, and a target vector,
    make a prediction and compute the gradient of the squared error
    loss with respect to the neuron weights.
    """
    # forward pass
    hidden_outputs, outputs = feed_forward(network, input_vector)

    # gradients with respect to output neuron pre-activation outputs
    output_deltas = [output * (1 - output) * (output - target)
                     for output, target in zip(outputs, target_vector)]

    # gradients with respect to output neuron weights
    output_grads = [[output_deltas[i] * hidden_output
                     for hidden_output in hidden_outputs + [1]]
                    for i, output_neuron in enumerate(network[-1])]

    # gradients with respect to hidden neuron pre-activation outputs
    hidden_deltas = [hidden_output * (1 - hidden_output) *
                         dot(output_deltas, [n[i] for n in network[-1]])
                     for i, hidden_output in enumerate(hidden_outputs)]

    # gradients with respect to hidden neuron weights
    hidden_grads = [[hidden_deltas[i] * input for input in input_vector + [1]]
                    for i, hidden_neuron in enumerate(network[0])]

    return [hidden_grads, output_grads]

In [87]:
# We’ll start by generating the training data and initializing our neural network with random weights
import random
random.seed(0)

# training data
xs = [[0., 0], [0., 1], [1., 0], [1., 1]]
ys = [[0.], [1.], [1.], [0.]]

# start with random weights
network = [ # hidden layer: 2 inputs -> 2 outputs
            [[random.random() for _ in range(2 + 1)],   # 1st hidden neuron
             [random.random() for _ in range(2 + 1)]],  # 2nd hidden neuron
            # output layer: 2 inputs -> 1 output
            [[random.random() for _ in range(2 + 1)]]   # 1st output neuron
          ]


In [88]:


# replace scratch.gradient_descent.gradient_step
def gradient_step(v, gradient, step_size):
    return [v_i + step_size * grad_i for v_i, grad_i in zip(v, gradient)]

import tqdm

learning_rate = 1.0

for epoch in tqdm.trange(20000, desc="neural net for xor"):
    for x, y in zip(xs, ys):
        gradients = sqerror_gradients(network, x, y)

        # update each layer
        network = [
            [gradient_step(neuron, grad, -learning_rate)
             for neuron, grad in zip(layer, layer_grad)]
            for layer, layer_grad in zip(network, gradients)
        ]

# check that it learned XOR
assert feed_forward(network, [0, 0])[-1][0] < 0.01
assert feed_forward(network, [0, 1])[-1][0] > 0.99
assert feed_forward(network, [1, 0])[-1][0] > 0.99
assert feed_forward(network, [1, 1])[-1][0] < 0.01

print("XOR training complete ✔")

neural net for xor: 100%|██████████| 20000/20000 [00:00<00:00, 32238.46it/s]

XOR training complete ✔


### Example: Fizz Buzz
Print the numbers 1 to 100, except that if the number is divisible
by 3, print "fizz"; if the number is divisible by 5, print "buzz";
and if the number is divisible by 15, print "fizzbuzz".

In [89]:
def fizz_buzz_encode(x: int) -> Vector:
    if x % 15 == 0:
        return [0, 0, 0, 1]
    elif x % 5 == 0:
        return [0, 0, 1, 0]
    elif x % 3 == 0:
        return [0, 1, 0, 0]
    else:
        return [1, 0, 0, 0]

assert fizz_buzz_encode(2) == [1, 0, 0, 0]
assert fizz_buzz_encode(6) == [0, 1, 0, 0]
assert fizz_buzz_encode(10) == [0, 0, 1, 0]
assert fizz_buzz_encode(30) == [0, 0, 0, 1]

In [90]:
def binary_encode(x: int) -> Vector:
    binary: List[float] = []

    for i in range(10):
        binary.append(x % 2)
        x = x // 2

    return binary

#                             1  2  4  8 16 32 64 128 256 512
assert binary_encode(0)   == [0, 0, 0, 0, 0, 0, 0, 0,  0,  0]
assert binary_encode(1)   == [1, 0, 0, 0, 0, 0, 0, 0,  0,  0]
assert binary_encode(10)  == [0, 1, 0, 1, 0, 0, 0, 0,  0,  0]
assert binary_encode(101) == [1, 0, 1, 0, 0, 1, 1, 0,  0,  0]
assert binary_encode(999) == [1, 1, 1, 0, 0, 1, 1, 1,  1,  1]

In [91]:
xs = [binary_encode(n) for n in range(101, 1024)]
ys = [fizz_buzz_encode(n) for n in range(101, 1024)]

In [92]:
# let’s create a neural network with random initial weights
NUM_HIDDEN = 25

network = [
    # hidden layer: 10 inputs -> NUM_HIDDEN outputs
    [[random.random() for _ in range(10 + 1)] for _ in range(NUM_HIDDEN)],

    # output_layer: NUM_HIDDEN inputs -> 4 outputs
    [[random.random() for _ in range(NUM_HIDDEN + 1)] for _ in range(4)]
]

In [93]:
# ===== PATCHED FIZZ BUZZ TRAINING (NO scratch) =====

# replace scratch.linear_algebra.squared_distance
def squared_distance(v, w):
    return sum((v_i - w_i) ** 2 for v_i, w_i in zip(v, w))

learning_rate = 1.0

with tqdm.trange(500) as t:
    for epoch in t:
        epoch_loss = 0.0

        for x, y in zip(xs, ys):
            predicted = feed_forward(network, x)[-1]
            epoch_loss += squared_distance(predicted, y)
            gradients = sqerror_gradients(network, x, y)

            network = [
                [gradient_step(neuron, grad, -learning_rate)
                 for neuron, grad in zip(layer, layer_grad)]
                for layer, layer_grad in zip(network, gradients)
            ]

        t.set_description(f"fizz buzz (loss: {epoch_loss:.2f})")

fizz buzz (loss: 29.62): 100%|██████████| 500/500 [01:00<00:00,  8.23it/s] 


In [94]:
# Our network will produce a four-dimensional vector of numbers, but we want a single prediction. 
#We’ll do that by taking the argmax, which is the index of the largest value:
def argmax(xs: list) -> int:
    """Returns the index of the largest value"""
    return max(range(len(xs)), key=lambda i: xs[i])

assert argmax([0, -1]) == 0               # items[0] is largest
assert argmax([-1, 0]) == 1               # items[1] is largest
assert argmax([-1, 10, 5, 20, -3]) == 3   # items[3] is largest

In [95]:
# finally we will solve this problem now:
num_correct = 0

for n in range(1, 101):
    x = binary_encode(n)
    predicted = argmax(feed_forward(network, x)[-1])
    actual = argmax(fizz_buzz_encode(n))
    labels = [str(n), "fizz", "buzz", "fizzbuzz"]
    print(n, labels[predicted], labels[actual])

    if predicted == actual:
        num_correct += 1

print(num_correct, "/", 100)

1 1 1
2 2 2
3 fizz fizz
4 4 4
5 buzz buzz
6 fizz fizz
7 7 7
8 8 8
9 fizz fizz
10 buzz buzz
11 11 11
12 fizz fizz
13 13 13
14 14 14
15 fizzbuzz fizzbuzz
16 16 16
17 17 17
18 fizz fizz
19 19 19
20 20 buzz
21 fizz fizz
22 22 22
23 23 23
24 fizz fizz
25 buzz buzz
26 26 26
27 fizz fizz
28 28 28
29 29 29
30 fizzbuzz fizzbuzz
31 31 31
32 32 32
33 fizz fizz
34 34 34
35 buzz buzz
36 fizz fizz
37 37 37
38 38 38
39 fizz fizz
40 buzz buzz
41 41 41
42 fizz fizz
43 43 43
44 44 44
45 fizzbuzz fizzbuzz
46 46 46
47 47 47
48 fizz fizz
49 49 49
50 buzz buzz
51 fizz fizz
52 52 52
53 53 53
54 fizz fizz
55 buzz buzz
56 56 56
57 fizz fizz
58 58 58
59 59 59
60 fizzbuzz fizzbuzz
61 61 61
62 62 62
63 fizz fizz
64 64 64
65 buzz buzz
66 fizz fizz
67 67 67
68 68 68
69 fizz fizz
70 buzz buzz
71 71 71
72 fizz fizz
73 73 73
74 74 74
75 fizzbuzz fizzbuzz
76 76 76
77 77 77
78 fizz fizz
79 79 79
80 80 buzz
81 fizz fizz
82 82 82
83 83 83
84 fizz fizz
85 fizz buzz
86 86 86
87 fizz fizz
88 88 88
89 89 89
90 fizzbuzz fizzbu

### Home work practice in pair
1. Please explore and understand the decision tree theory and code<br>
2. (6 points) Please work on Iris dataset using neural network you learned in this lecture, and try to make predictions for:<br>
<br>['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
<br>[6.0, 3.0, 5.0, 0.6]
<br>[6.0, 3.0, 5.0, 1.6]
<br>[6.0, 3.0, 5.0, 2.6]
3. (4 points) Please explore neural network on sklearn and train it on Iris dataset and make predictions on the three samples above.

In [96]:
from sklearn.datasets import load_iris
import numpy as np

iris = load_iris()
X = iris.data
y = iris.target

# one-hot encoding for neural network
def one_hot(y, num_classes):
    result = np.zeros((len(y), num_classes))
    for i, val in enumerate(y):
        result[i][val] = 1
    return result

y_encoded = one_hot(y, 3)

In [97]:
import math
import random

# sigmoid activation
def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def dot(v, w):
    return sum(v_i * w_i for v_i, w_i in zip(v, w))

# initialize weights (small random)
random.seed(0)
input_size = 4
hidden_size = 5
output_size = 3

# weights
hidden_weights = [[random.random() for _ in range(input_size)] for _ in range(hidden_size)]
hidden_bias = [random.random() for _ in range(hidden_size)]

output_weights = [[random.random() for _ in range(hidden_size)] for _ in range(output_size)]
output_bias = [random.random() for _ in range(output_size)]

# forward pass
def predict_nn(x):
    hidden = [sigmoid(dot(w, x) + b) for w, b in zip(hidden_weights, hidden_bias)]
    output = [sigmoid(dot(w, hidden) + b) for w, b in zip(output_weights, output_bias)]
    return output

In [98]:
samples = [
    [6.0, 3.0, 5.0, 0.6],
    [6.0, 3.0, 5.0, 1.6],
    [6.0, 3.0, 5.0, 2.6]
]

print("Manual Neural Network Predictions:")

for s in samples:
    out = predict_nn(s)
    pred = np.argmax(out)
    print(s, "->", iris.target_names[pred])

Manual Neural Network Predictions:
[6.0, 3.0, 5.0, 0.6] -> virginica
[6.0, 3.0, 5.0, 1.6] -> virginica
[6.0, 3.0, 5.0, 2.6] -> virginica


In [99]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

# split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# train model
model = MLPClassifier(hidden_layer_sizes=(5,), max_iter=500)
model.fit(X_train, y_train)

/Users/songsidiya/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


MLPClassifier(hidden_layer_sizes=(5,), max_iter=500)

In [100]:
print("\nSklearn Neural Network Predictions:")

predictions = model.predict(samples)

for s, p in zip(samples, predictions):
    print(s, "->", iris.target_names[p])


Sklearn Neural Network Predictions:
[6.0, 3.0, 5.0, 0.6] -> versicolor
[6.0, 3.0, 5.0, 1.6] -> virginica
[6.0, 3.0, 5.0, 2.6] -> virginica


In [101]:
from sklearn.datasets import load_iris
data = load_iris()

In [102]:
print(data.data)
print(data.target)

[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]
 [5.4 3.9 1.7 0.4]
 [4.6 3.4 1.4 0.3]
 [5.  3.4 1.5 0.2]
 [4.4 2.9 1.4 0.2]
 [4.9 3.1 1.5 0.1]
 [5.4 3.7 1.5 0.2]
 [4.8 3.4 1.6 0.2]
 [4.8 3.  1.4 0.1]
 [4.3 3.  1.1 0.1]
 [5.8 4.  1.2 0.2]
 [5.7 4.4 1.5 0.4]
 [5.4 3.9 1.3 0.4]
 [5.1 3.5 1.4 0.3]
 [5.7 3.8 1.7 0.3]
 [5.1 3.8 1.5 0.3]
 [5.4 3.4 1.7 0.2]
 [5.1 3.7 1.5 0.4]
 [4.6 3.6 1.  0.2]
 [5.1 3.3 1.7 0.5]
 [4.8 3.4 1.9 0.2]
 [5.  3.  1.6 0.2]
 [5.  3.4 1.6 0.4]
 [5.2 3.5 1.5 0.2]
 [5.2 3.4 1.4 0.2]
 [4.7 3.2 1.6 0.2]
 [4.8 3.1 1.6 0.2]
 [5.4 3.4 1.5 0.4]
 [5.2 4.1 1.5 0.1]
 [5.5 4.2 1.4 0.2]
 [4.9 3.1 1.5 0.2]
 [5.  3.2 1.2 0.2]
 [5.5 3.5 1.3 0.2]
 [4.9 3.6 1.4 0.1]
 [4.4 3.  1.3 0.2]
 [5.1 3.4 1.5 0.2]
 [5.  3.5 1.3 0.3]
 [4.5 2.3 1.3 0.3]
 [4.4 3.2 1.3 0.2]
 [5.  3.5 1.6 0.6]
 [5.1 3.8 1.9 0.4]
 [4.8 3.  1.4 0.3]
 [5.1 3.8 1.6 0.2]
 [4.6 3.2 1.4 0.2]
 [5.3 3.7 1.5 0.2]
 [5.  3.3 1.4 0.2]
 [7.  3.2 4.7 1.4]
 [6.4 3.2 4.5 1.5]
 [6.9 3.1 4.

In [103]:
def inputRecord(inputV):
    resultVector = []
    for i in range(len(inputV)):
        if inputV[i] < 3:
            resultVector+=[0,0,1]
        elif inputV[i] < 6:
            resultVector+=[0,1,0]
        else:
            resultVector+=[1,0,0]
    return resultVector
print(inputRecord([6.8,3.2,5.9,2.3]))

def outputRecord(label):
    if label == 0:
        return [0,0,1]
    elif label==1:
        return [0,1,0]
    else:
        return [1,0,0]


[1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1]
